In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory

# 다운 받은 데이터 path를 만들어 두고요,
PATH = os.path.join(os.getcwd(), '/content/drive/MyDrive/최강<포빅아A4>/AI프로젝트/견인/데이터')

# train, validation, test 폴더 이름도 정해주고요.
train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')
test_dir = os.path.join(PATH, 'test')  # test 폴더 경로 추가

# image_dataset_from_directory를 이용하면 batch size와 image size에 맞춰서 자동으로 나눠줍니다.
BATCH_SIZE = 32
IMG_SIZE = (160, 160)

train_dataset = image_dataset_from_directory(train_dir,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             label_mode='categorical')

validation_dataset = image_dataset_from_directory(validation_dir,
                                                  shuffle=True,
                                                  batch_size=BATCH_SIZE,
                                                  image_size=IMG_SIZE,
                                                  label_mode='categorical')

test_dataset = image_dataset_from_directory(test_dir,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             label_mode='categorical')  # 원-핫 인코딩된 레이블을 위해

Found 1705 files belonging to 4 classes.
Found 244 files belonging to 4 classes.
Found 487 files belonging to 4 classes.


In [ ]:
AUTOTUNE = tf.data.experimental.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

In [ ]:
layer_augmentation = tf.keras.Sequential([
  tf.keras.layers.experimental.preprocessing.RandomFlip('horizontal'),
  tf.keras.layers.experimental.preprocessing.RandomRotation(0.2),
])

In [ ]:
layer_monet_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [ ]:
layer_rescale = tf.keras.layers.experimental.preprocessing.Rescaling(1./127.5, offset= -1)

In [ ]:
import tensorflow as tf

IMG_SHAPE = (160, 160, 3)

# MobileNetV3 Small 모델 불러오기
model_base = tf.keras.applications.MobileNetV3Small(input_shape=IMG_SHAPE,
                                                    include_top=False,
                                                    weights='imagenet')
model_base.trainable = False  # 기본 모델의 가중치를 고정


4334752/4334752 [==============================] - 1s 0us/step


In [ ]:
# 데이터 증강 및 전처리 층
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.experimental.preprocessing.RandomFlip('horizontal'),
    tf.keras.layers.experimental.preprocessing.RandomRotation(0.2),
])

# 모델 구성
inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = model_base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)  # 과적합 방지를 위한 Dropout 층
outputs = tf.keras.layers.Dense(4, activation='softmax')(x)  # 클래스가 4개인 경우

model = tf.keras.Model(inputs, outputs)


In [ ]:
base_learning_rate = 0.0001
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
              loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

# 모델 학습
# `train_dataset`과 `validation_dataset`은 이미 정의되어 있어야 합니다.
history = model.fit(train_dataset,
                    epochs=10,
                    validation_data=validation_dataset)


Epoch 1/10
54/54 [==============================] - 489s 8s/step - loss: 1.9217 - accuracy: 0.2481 - val_loss: 1.6576 - val_accuracy: 0.2500
Epoch 2/10
54/54 [==============================] - 22s 366ms/step - loss: 1.5792 - accuracy: 0.2768 - val_loss: 1.4235 - val_accuracy: 0.2582
Epoch 3/10
54/54 [==============================] - 21s 342ms/step - loss: 1.3706 - accuracy: 0.3537 - val_loss: 1.2838 - val_accuracy: 0.3443
Epoch 4/10
54/54 [==============================] - 22s 339ms/step - loss: 1.2085 - accuracy: 0.4622 - val_loss: 1.1607 - val_accuracy: 0.4426
Epoch 5/10
54/54 [==============================] - 22s 363ms/step - loss: 1.1034 - accuracy: 0.5196 - val_loss: 1.0536 - val_accuracy: 0.5451
Epoch 6/10
54/54 [==============================] - 21s 342ms/step - loss: 0.9754 - accuracy: 0.5883 - val_loss: 0.9614 - val_accuracy: 0.6270
Epoch 7/10
54/54 [==============================] - 22s 338ms/step - loss: 0.9074 - accuracy: 0.6387 - val_loss: 0.8847 - val_accuracy: 0.6721
E